In [17]:
import pandas as pd
from datetime import datetime

# 1. Configuration variables with absolute paths
COUNTRIES = ["US", "GB", "IN"]
FILE_PATHS = {
    "US": "/data/USvideos.csv",
    "GB": "/data/GBvideos.csv",
    "IN": "/data/INvideos.csv",
    "categories": "/data/categories.csv"
}
VIEW_THRESHOLD = 1_000_000

# 2. Formatted run header using f-strings
print(f"==========================================")
print(f"=== TrendPulse Part 1 Execution Header ===")
print(f"Target Countries : {', '.join(COUNTRIES)}")
print(f"Execution Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"==========================================")

=== TrendPulse Part 1 Execution Header ===
Target Countries : US, GB, IN
Execution Started: 2026-11-09 13:14:41


### Notebook Discipline Note
This notebook is structured with clear headings, designed to run top-to-bottom without errors via *Runtime -> Restart and run all*, and contains no leftover scratch cells.

In [20]:
def load_country(path, country_code, max_rows=None):
    """
    Reusable function to load a country CSV, add a country tag,
    and return records as a list of dictionaries.
    """
    df = pd.read_csv(path, nrows=max_rows)
    df["country"] = country_code
    records = df.to_dict("records")
    print(f"Successfully loaded {len(records):,} records for country: {country_code}")
    return records

# Load records for all three countries into a single master list
all_records = []
for code in COUNTRIES:
    path = f"/data/{code}videos.csv"
    records = load_country(path, code)
    all_records.extend(records)

print(f"\nTotal combined records loaded: {len(all_records):,}")

# Build category lookup dictionary from categories.csv using the correct columns
category_names = dict(zip(categories_df['category_id'].astype(str), categories_df['category_name']))
print(f"Loaded {len(category_names)} category mappings successfully!")


Successfully loaded 16,080 records for country: US
Successfully loaded 15,075 records for country: GB
Successfully loaded 15,075 records for country: IN

Total combined records loaded: 46,230
Loaded 17 category mappings successfully!


In [21]:
cleaned_records = []
error_count = 0

for record in all_records:
    try:
        cleaned_record = {
            "video_id": str(record.get("video_id", "")),
            "title": str(record.get("title", "")),
            "channel_title": str(record.get("channel_title", "")),
            "category_id": str(record.get("category_id", "")),
            "views": int(record.get("views", 0)),
            "likes": int(record.get("likes", 0)),
            "dislikes": int(record.get("dislikes", 0)),
            "comment_count": int(record.get("comment_count", 0)),
            "country": record.get("country", "Unknown")
        }

        # Add the human-readable category name using our lookup dictionary
        cat_id = cleaned_record["category_id"]
        cleaned_record["category_name"] = category_names.get(cat_id, "Unknown Category")

        cleaned_records.append(cleaned_record)

    except (ValueError, TypeError) as e:
        error_count += 1

print(f"Data cleaning complete!")
print(f"Successfully cleaned records: {len(cleaned_records):,}")
print(f"Skipped malformed rows due to errors: {error_count:,}")

Data cleaning complete!
Successfully cleaned records: 46,182
Skipped malformed rows due to errors: 48


In [22]:
import pandas as pd

# Convert cleaned records to a DataFrame for analysis
df_clean = pd.DataFrame(cleaned_records)

# 1. Filter videos meeting the view threshold (1,000,000 views)
popular_videos = df_clean[df_clean["views"] >= VIEW_THRESHOLD]
print(f"Total videos with >= {VIEW_THRESHOLD:,} views: {len(popular_videos):,}")

# 2. Find the top 5 most viewed trending videos across all regions
top_5_videos = df_clean.nlargest(5, "views")[["title", "channel_title", "views", "country"]]
print("\n--- Top 5 Most Viewed Trending Videos ---")
display(top_5_videos)

# 3. Quick summary statistics by country
country_summary = df_clean.groupby("country").agg(
    total_videos=("video_id", "count"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean")
).reset_index()

print("\n--- Summary Statistics by Country ---")
display(country_summary)

Total videos with >= 1,000,000 views: 9,727

--- Top 5 Most Viewed Trending Videos ---


,title,channel_title,views,country
26498,Lena Ortiz - Monsoon (Live),FinalSpeech81,360760164,GB
30922,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,343297002,GB
26297,Lena Ortiz - Monsoon (Live),FinalSpeech81,286714357,GB
30721,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,282812154,GB
24084,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,255362869,GB



--- Summary Statistics by Country ---


,country,total_videos,avg_views,avg_likes
0,GB,15059,1.581263e+06,65134.136662
1,IN,15059,9.544438e+05,33217.693207
2,US,16064,1.012406e+06,39509.171439
